In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Unzip reviews parquet
!unzip "/content/drive/MyDrive/Copy of reviews_eda_sample.parquet-20251009T224454Z-1-001.zip" -d "/content/"

# Unzip meta parquet
!unzip "/content/drive/MyDrive/Copy of meta.parquet-20251009T224452Z-1-001.zip" -d "/content/"

# Verify extraction
!ls /content/

In [ ]:
!pip install pyspark -q

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("AmazonReviews")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "32")
    .getOrCreate()
)

# Load extracted parquet data
reviews_raw = spark.read.parquet("/content/reviews_eda_sample.parquet")
meta = spark.read.parquet("/content/meta.parquet")

print(f"Reviews: {reviews_raw.count():,}")
print(f"Meta: {meta.count():,}")

In [ ]:
# Step 1: Rename conflicting columns before join
reviews_renamed = reviews_raw.withColumnRenamed("title", "review_title") \
                             .withColumnRenamed("images", "review_images")

meta_renamed = meta.withColumnRenamed("title", "product_title") \
                   .withColumnRenamed("images", "product_images")

# Step 2: Join reviews with metadata
joined = reviews_renamed.join(meta_renamed, on="parent_asin", how="left")

# Step 3: Filter Electronics category only
reviews_elec = joined.filter(
    (F.col("main_category") == "Electronics") |
    (F.array_contains(F.col("categories"), "Electronics"))
)

# Step 4: Select columns and add derived features
reviews_clean = reviews_elec.select(
    "parent_asin", "asin", "user_id", "rating", "text",
    "review_title", "timestamp", "helpful_vote", "verified_purchase",
    "review_images", "main_category", "store", "average_rating", "rating_number"
).withColumn(
    "has_image", (F.size(F.col("review_images")) > 0).cast("int")
).withColumn(
    "review_len", F.length(F.col("text"))
).withColumn(
    "review_date", F.from_unixtime(F.col("timestamp") / 1000).cast("date")
).withColumn(
    "verified_int", F.when(F.col("verified_purchase") == True, 1).otherwise(0)
)

# Step 5: Remove invalid rows
reviews_clean = reviews_clean.filter(
    (F.col("text").isNotNull()) &
    (F.col("user_id").isNotNull()) &
    (F.col("rating").isNotNull())
)

reviews_clean.cache()
print(f"Cleaned reviews: {reviews_clean.count():,}")
reviews_clean.printSchema()

In [ ]:
# Save cleaned dataset to Drive for other tasks
output_path = "/content/drive/MyDrive/reviews_cleaned.parquet"
reviews_clean.write.mode("overwrite").parquet(output_path)
print(f"Saved to: {output_path}")

In [ ]:
# Verify the saved file
!ls -la /content/drive/MyDrive/reviews_cleaned.parquet/ | head -10

# Quick test load
test_df = spark.read.parquet("/content/drive/MyDrive/reviews_cleaned.parquet")
print(f"Verified rows: {test_df.count():,}")
test_df.show(5)